**Дисперсія** - ступінь розсіювання навколо середнього значення випадкової величини. У
статистичному розумінні дисперсія є середнє арифметичне із квадратів відхилень величин від їх
середнього арифметичного


**Дисперсійний аналіз** – це сукупність методів, які дозволяють перевіряти: чи змінюється
середнє значення характеристик деяких об’єктів в залежності від дії деякого фактора або
кількох факторів. Він дає змогу визначити вплив одного або декількох факторів на
результативний показник.

Можуть враховуватись як кількісні та і якісні ознаки!!!

Розрізняють однофакторний та багатофакторний дисперсійний аналіз


# Приклад №1--однофакторний дисперсійний аналіз
Кожний співак Нью-Йоркської Хорової спільноти в 1979 році офіційно доповідав про свій зріст. Голоси співаків поділяються від найнижчого за тоном до найвищого на такі типи: сопрано, альт, тенор, бас. Зазвичай 2 перші типи належать жінкам, 2 останні – чоловікам.
Потрібно дослідити – як зріст впливає на голосовий діапазон співака чи співачки. Також потрібно зробити порівняння сопрано та альтів і, окремо, порівняння тенорів і басів.


Дані опитування містяться в файлі voiсe.txt, що має наступну структуру:<br>
Hei	Factor<br>
63	Soprano<br>
61	Alto<br>
67	Tenor<br>
70	Bass<br>
................... <br>
Для проведення аналізу спочатку занесемо дані  у  фрейм  з  двома  стовпчиками-змінними:  один  відповідає  зросту  людини  (**Hei**,  залежна  змінна),  другий – тип голосу (**Factor**, фактор):

In [ ]:
# імпорт бібліотеки для роботи із даними та аналізом
import pandas as pd
import gdown

google_drive_url = 'https://drive.google.com/file/d/1zc-C6y8EyHlhAYtLZ4fhwq_baA6IBDSB'

file_id = google_drive_url.split('/')[-1]

gdown.download(f'https://drive.google.com/uc?id={file_id}', 'Voice.txt', quiet=False)

# Зчитування даних з файлу
input_data = pd.read_csv('/content/Voice.txt', delimiter='\t')

print(input_data)

Downloading...
From: https://drive.google.com/uc?id=1zc-C6y8EyHlhAYtLZ4fhwq_baA6IBDSB
To: /content/Voice.txt
100%|██████████| 1.32k/1.32k [00:00<00:00, 3.36MB/s]

     Hei   Factor
0     64  Soprano
1     62  Soprano
2     66  Soprano
3     65  Soprano
4     60  Soprano
..   ...      ...
125   75     Bass
126   68     Bass
127   71     Bass
128   70     Bass
129   74     Bass

[130 rows x 2 columns]


Щоб запустити програму однофакторного дисперсійного аналізу Python, скористаємось комбінацією функцій **ols()**, **fit()** та **anova_lm()** (скорочення від **ANOVA** – analisis of variances). В аргументах вказуємо спершу залежну змінну, потім – фактор.

In [ ]:
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

# Проведення однофакторного дисперсійного аналізу
model = ols('Hei ~ Factor', data=input_data).fit()
anova_result = anova_lm(model)

print(anova_result)

             df       sum_sq     mean_sq          F        PR(>F)
Factor      3.0  1046.231343  348.743781  54.313546  1.334470e-22
Residual  126.0   809.037888    6.420936        NaN           NaN


Якщо p-значення менше за заданий рівень значущості (часто використовується рівень **0.05**), то ми вважаємо результат статистично значущим.<br>
У даному випадку p-значення дуже мале (**1.334470e-22**), що значно менше за 0.05, отже, ми можемо вважати, що фактор **"Factor"** суттєво впливає на змінну **"Hei"**.<br>
Отже, на рівні значущості **0.05** було досягнуто статистично значущого впливу фактора.


Але серед співаків у вибірці є і жінки, і чоловіки. Може, ефект залежності просто показує відмінність між жінками і чоловіками – жінки в середньому нижчі і їх тембр в середньому вищий. Чи буде така сама залежність всередині  статі? <br>
Вилучимо з вибірки чоловіків, для цього створимо новий фрейм даних, де змінна Factor не може приймати значення **Bass** і **Tenor**.  


In [ ]:
# Фільтрація даних за умовою, що Factor не може приймати значення 'Bass' або 'Tenor'
filtered_data_women = input_data[~input_data['Factor'].isin(['Bass', 'Tenor'])]
print(filtered_data_women)

    Hei   Factor
0    64  Soprano
1    62  Soprano
2    66  Soprano
3    65  Soprano
4    60  Soprano
..  ...      ...
67   64     Alto
68   67     Alto
69   66     Alto
70   68     Alto
71   66     Alto

[72 rows x 2 columns]


Запускаємо програму однофакторного дисперсійного аналізу згідно умов вибірки.  

In [ ]:
# Проведення однофакторного дисперсійного аналізу
model = ols('Hei ~ Factor', data=filtered_data_women).fit()
anova_result = anova_lm(model)

print(anova_result)

            df  sum_sq   mean_sq         F    PR(>F)
Factor     1.0     8.0  8.000000  1.437741  0.234549
Residual  70.0   389.5  5.564286       NaN       NaN


Результат:<br>
Досягнутий рівень значущості **р = 0.234549**, що значно більше звичайної порогової величини альфа. Отже, серед жінок відсутній зв’язок між  зростом і тембром голосу.

Вилучимо з вибірки чоловіків, для цього створимо новий фрейм даних, де змінна Factor не може приймати значення **Bass** і **Tenor**.



In [ ]:
# Фільтрація даних за умовою, що Factor не може приймати значення 'Alto' або 'Soprano'
filtered_data_men = input_data[~input_data['Factor'].isin(['Alto', 'Soprano'])]

Проведемо однофакторноний дисперсійний аналіз серед чоловіків:

In [ ]:
# Проведення однофакторного дисперсійного аналізу
model = ols('Hei ~ Factor', data=filtered_data_men).fit()
anova_result = anova_lm(model)

print(anova_result)

            df      sum_sq   mean_sq         F    PR(>F)
Factor     1.0    3.582801  3.582801  0.478233  0.492081
Residual  56.0  419.537888  7.491748       NaN       NaN


Згідно результату  **р = 0,492081**, що також що значно більше звичайної порогової величини альфа.<br>
Отже, серед чоловіків також відсутній зв’язок між зростом і тембром.

# Приклад №2 -- Двофакторний дисперсійний аналіз

Потрібно дослідити,  чи  присутня залежність між кількістю помилок, яких  припускаються  щури при проходженні лабіринту, від їх оточуючого  середовища та від генетичних  властивостей.



 Дані експерименту занесені у файл  rat.txt, що має наступну структуру <br>
ENV. ERR. STR. <br>
Free 26 Bright<br>
Free 14 Bright<br>
Free 41 Bright<br>
Free 16 Bright<br>
Free 41 Mixed<br>
Free 82 Mixed<br>

Назви змінних:  
1. **ENVIRONMET** = Оточуюче середовище: FREE – вільне; RESTRCTЕD - обмежене.  
2. **STRAIN** = Генетична характеристика: BRIGHT – чиста лінія «розумних» пацюків;   MIXED - мішаної крові; DULL– чиста лінія «дурних» пацюків.
3. **ERRORS** = Кількість помилок, яких припустився щур при проходженні лабіринту.


Занесемо дані у фрейм з трьома  стовпчиками-змінними: ERRORS – залежна змінна; ENVIRONMENT,  STRAIN – фактори  та запускаємо програму  дисперсійного  аналізу. В аргументах  вказуємо,  що  перевіряємо вплив і середовища, і генетичних властивостей, і сукупності дій цих факторів.  

In [ ]:
import pandas as pd
import gdown

google_drive_url = 'https://drive.google.com/file/d/14qreu8glN6Yq9w1ECYYJy2BlpD_OAi0I'

file_id = google_drive_url.split('/')[-1]

gdown.download(f'https://drive.google.com/uc?id={file_id}', 'Rat.txt', quiet=False)

# Зчитування даних з файлу
input_data_rat = pd.read_csv('/content/Rat.txt', delim_whitespace=True)

print(input_data_rat)

Downloading...
From: https://drive.google.com/uc?id=14qreu8glN6Yq9w1ECYYJy2BlpD_OAi0I
To: /content/Rat.txt
100%|██████████| 459/459 [00:00<00:00, 1.27MB/s]

     ENVIRNMNT  ERRORS  STRAIN
0         Free      26  Bright
1         Free      14  Bright
2         Free      41  Bright
3         Free      16  Bright
4         Free      41   Mixed
5         Free      82   Mixed
6         Free      26   Mixed
7         Free      86   Mixed
8         Free      36    Dull
9         Free      87    Dull
10        Free      39    Dull
11        Free      99    Dull
12  Restricted      51  Bright
13  Restricted      35  Bright
14  Restricted      96  Bright
15  Restricted      36  Bright
16  Restricted      39   Mixed
17  Restricted     114   Mixed
18  Restricted     104   Mixed
19  Restricted      92   Mixed
20  Restricted      42    Dull
21  Restricted     133    Dull
22  Restricted      92    Dull
23  Restricted     124    Dull


Запускаємо програму двофакторного дисперсійного аналізу вибірки.

In [ ]:
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

# Виконуємо двофакторний дисперсійний аналіз
model = ols('ERRORS ~ ENVIRNMNT + STRAIN + ENVIRNMNT:STRAIN', input_data_rat).fit()
anova_table = anova_lm(model, typ=2)

print(anova_table)

                        sum_sq    df         F    PR(>F)
ENVIRNMNT          5551.041667   1.0  5.822516  0.026705
STRAIN             7939.750000   2.0  4.164023  0.032635
ENVIRNMNT:STRAIN     16.083333   2.0  0.008435  0.991604
Residual          17160.750000  18.0       NaN       NaN


Згідно результату, гіпотезу про рівність середніх при зміні фактора середовища  ENVIRONMENT  відхиляємо, бо **р = 0,026705** - вплив є. Аналогічно спостерігається вплив генетичного фактора STRAIN (**р = 0,03264**). А от впливу комбінації цих факторів нема: **р = 0,9916**.